In [1]:
!pip install -U sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer

c:\Users\detix\anaconda3\envs\tii\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4411.63it/s]


In [ ]:

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium."
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

torch.Size([3, 3])


In [5]:
!pip install -U fastapi uvicorn typing-extensions pyngrok requests

In [6]:
from fastapi import FastAPI, Body
from fastapi.responses import StreamingResponse, JSONResponse
from io import BytesIO
import zipfile
import io
import gc
import torch

app = FastAPI(title="Embedding API", port=7895)

@app.get("/")
def health():
    return {"status": "ok", "message": "API is running"}

@app.post("/embeddings")
def embedding(text: str = Body(..., embed=True)):
    embedding = model.encode(text)
    return JSONResponse(content={"embedding": embedding.tolist()})

In [ ]:
from pyngrok import ngrok
import threading
import uvicorn
from env import NGROK_AUTH_TOKEN

# add your ngrok token once
ngrok.set_auth_token("")

PORT = 8004

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url if hasattr(tunnel, "public_url") else str(tunnel)

print("Public API URL:", public_url)
print("POST to:", f"{public_url}/embeddings")

INFO:     Started server process [28928]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8004 (Press CTRL+C to quit)


Public API URL: https://7dbb-62-44-108-4.ngrok-free.app
POST to: https://7dbb-62-44-108-4.ngrok-free.app/embeddings


INFO:     62.44.108.4:0 - "POST /embeddings HTTP/1.1" 200 OK


In [8]:
import requests

base_url = public_url.public_url if hasattr(public_url, "public_url") else str(public_url)
response = requests.post(
    f"{base_url}/embeddings",
    json={"text": "The weather is lovely today."}
 )
print(response.json())

{'embedding': [-0.0390625, 0.01611328125, -0.004730224609375, 0.00811767578125, 0.053466796875, -0.0216064453125, 0.0220947265625, 0.0003376007080078125, -0.07080078125, -0.006011962890625, 0.003875732421875, 0.08056640625, 0.00970458984375, -0.005828857421875, -0.04541015625, 0.08154296875, 0.01336669921875, 0.09033203125, 0.1337890625, -0.072265625, 0.0252685546875, -0.02294921875, -0.0002956390380859375, 0.095703125, -0.01043701171875, 0.01446533203125, -0.000339508056640625, 0.062255859375, 0.051513671875, -0.009033203125, 0.032470703125, 0.0186767578125, -0.01202392578125, 0.00604248046875, -0.019287109375, -0.01116943359375, 0.052978515625, 0.0322265625, -0.016357421875, 0.031005859375, -0.020263671875, 0.03759765625, 0.0262451171875, 0.0208740234375, -0.0125732421875, -0.0244140625, 0.0260009765625, -0.0272216796875, -0.040283203125, -0.0162353515625, -0.02294921875, 0.01470947265625, -0.027587890625, -0.041015625, -0.0341796875, -0.04052734375, 0.005218505859375, 0.007202148437

In [14]:
import numpy as np

In [15]:
np.array(response.json()['embedding']).shape

(1024,)

In [9]:
from pyngrok import ngrok

# Disconnect all active ngrok tunnels and kill all ngrok processes
print("Killing all active ngrok processes...")
ngrok.kill()
print("All ngrok processes killed.")

Killing all active ngrok processes...
All ngrok processes killed.
